# Distancias de Wasserstein

Este cuaderno acompaña el **Capítulo 10** de las notas del curso (*Distancias de Wasserstein*). Exploramos computacionalmente:

- El cálculo de $W_p$ en **dimensión uno** mediante las pseudoinversas $F^{[-1]}$, comparado con la solución del programa lineal.
- La **fórmula cerrada para gaussianas** y su verificación por Monte Carlo.
- La comparación entre **distintos exponentes**: $W_q \le W_p$ para $q \le p$ y la desigualdad recíproca en soportes acotados.
- La relación entre **convergencia en $W_p$, convergencia débil y momentos**, con el ejemplo de la masa que se escapa.
- La **interpolación por desplazamiento** (geodésicas de McCann) frente a la interpolación lineal.
- La **dualidad de Kantorovich–Rubinstein** para $W_1$.

Utilizamos la biblioteca [POT (Python Optimal Transport)](https://pythonot.github.io/) para resolver el problema de Kantorovich exacto, y `scipy` para los programas lineales de la última sección.

In [ ]:
# @title
pip install POT

In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt
import ot
from scipy.stats import norm
from scipy.optimize import linprog
from scipy.linalg import sqrtm

rng = np.random.default_rng(0)

## 1. $W_p$ en dimensión uno

Recordemos la definición: para $\mu,\nu\in\mathcal P_p(\mathbb R^d)$,

$$
W_p(\mu,\nu)=\Bigl(\min_{\pi\in\Pi(\mu,\nu)}\int |x-y|^p\,d\pi\Bigr)^{1/p}.
$$

En dimensión uno el plan comonótono es óptimo para todo costo $h(x-y)$ con $h$ convexa, y en particular para $|x-y|^p$ con $p\ge1$. Esto da la fórmula

$$
W_p(\mu,\nu)^p=\int_0^1\bigl|F_\mu^{[-1]}(t)-F_\nu^{[-1]}(t)\bigr|^p\,dt ,
$$

que reduce el cálculo de $W_p$ a una integral en $[0,1]$. Para medidas empíricas con $n$ átomos de igual peso, la pseudoinversa es constante a trozos y la integral es una suma sobre los datos **ordenados**:

$$
W_p(\mu_n,\nu_n)^p=\frac1n\sum_{k=1}^n |x_{(k)}-y_{(k)}|^p .
$$

Verificamos esto contra la solución del programa lineal que calcula POT (`ot.emd2`), que no sabe nada de la estructura unidimensional.

In [ ]:
def Wp_1d(x, y, p):
    """W_p entre las medidas empíricas uniformes de las muestras x e y (mismo tamaño), vía cuantiles."""
    xs, ys = np.sort(x), np.sort(y)
    return np.mean(np.abs(xs - ys)**p)**(1/p)

def Wp_lp(x, y, p, a=None, b=None):
    """W_p vía el programa lineal (POT). Funciona en cualquier dimensión."""
    x = np.atleast_2d(x.T).T if x.ndim == 1 else x
    y = np.atleast_2d(y.T).T if y.ndim == 1 else y
    a = np.ones(len(x))/len(x) if a is None else a
    b = np.ones(len(y))/len(y) if b is None else b
    M = ot.dist(x, y, metric='euclidean')**p
    return ot.emd2(a, b, M, numItermax=10_000_000)**(1/p)

n = 300
x = rng.normal(0, 1, n)                 # muestra de N(0,1)
y = rng.exponential(1.5, n) - 1         # muestra de Exp(1/1.5) desplazada

print(f"{'p':>3} {'cuantiles':>12} {'LP (POT)':>12}")
for p in [1, 2, 3, 5]:
    print(f"{p:>3} {Wp_1d(x, y, p):>12.6f} {Wp_lp(x, y, p):>12.6f}")

Las dos columnas coinciden hasta la precisión del solver. El cálculo por cuantiles tiene costo $O(n\log n)$ (ordenar); el programa lineal, en el peor caso, $O(n^3\log n)$. Ilustramos la fórmula gráficamente: $W_1$ es el área entre las dos pseudoinversas, que coincide con el área entre las dos funciones de distribución.

In [ ]:
t = (np.arange(n) + 0.5)/n
xs, ys = np.sort(x), np.sort(y)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].step(t, xs, where='mid', label=r'$F_\mu^{[-1]}$')
ax[0].step(t, ys, where='mid', label=r'$F_\nu^{[-1]}$')
ax[0].fill_between(t, xs, ys, alpha=0.25, step='mid')
ax[0].set_xlabel('t'); ax[0].set_title(r'$W_1$ = área entre las pseudoinversas'); ax[0].legend()

grid = np.linspace(min(xs.min(), ys.min()), max(xs.max(), ys.max()), 800)
Fx = np.searchsorted(xs, grid, side='right')/n
Fy = np.searchsorted(ys, grid, side='right')/n
ax[1].plot(grid, Fx, label=r'$F_\mu$'); ax[1].plot(grid, Fy, label=r'$F_\nu$')
ax[1].fill_between(grid, Fx, Fy, alpha=0.25)
ax[1].set_xlabel('x'); ax[1].set_title(r'... = área entre las distribuciones'); ax[1].legend()
plt.tight_layout(); plt.show()

print("W_1 por cuantiles      :", Wp_1d(x, y, 1))
print("W_1 = ∫|F_mu - F_nu| dx:", np.trapezoid(np.abs(Fx - Fy), grid))

**Ejercicio.** La igualdad $W_1(\mu,\nu)=\int_{\mathbb R}|F_\mu(x)-F_\nu(x)|\,dx$ vale sólo para $p=1$. Verificar numéricamente que $\int|F_\mu-F_\nu|^2\,dx$ **no** es $W_2^2$, y explicar geométricamente por qué (las áreas entre las curvas coinciden, pero las integrales de las potencias no: en un caso se integra en $t$ y en el otro en $x$).

## 2. Gaussianas

Para $\mu=N(m_0,\Sigma_0)$ y $\nu=N(m_1,\Sigma_1)$ en $\mathbb R^d$ vale

$$
W_2(\mu,\nu)^2=|m_0-m_1|^2+\operatorname{tr}\Bigl(\Sigma_0+\Sigma_1-2\bigl(\Sigma_0^{1/2}\Sigma_1\Sigma_0^{1/2}\bigr)^{1/2}\Bigr),
$$

y el mapa óptimo es afín, $T(x)=m_1+A(x-m_0)$ con $A=\Sigma_0^{-1/2}(\Sigma_0^{1/2}\Sigma_1\Sigma_0^{1/2})^{1/2}\Sigma_0^{-1/2}$. Implementamos la fórmula y la comparamos con:

1. la función `ot.gaussian.bures_wasserstein_distance` de POT (misma fórmula, otra implementación);
2. una estimación **Monte Carlo**: $W_2$ entre medidas empíricas de $n$ muestras de cada gaussiana, resuelta por programación lineal.

La estimación Monte Carlo converge a $W_2(\mu,\nu)$ cuando $n\to\infty$, pero lentamente: la distancia entre una medida y su medida empírica en $\mathbb R^d$ decae como $n^{-1/d}$ para $d\ge3$ (y como $n^{-1/2}$, con correcciones logarítmicas, en dimensión baja).

In [ ]:
def W2_gauss(m0, S0, m1, S1):
    S0h = np.real(sqrtm(S0))
    C = np.real(sqrtm(S0h @ S1 @ S0h))
    return np.sqrt(np.sum((m0 - m1)**2) + np.trace(S0 + S1 - 2*C))

def mapa_gauss(m0, S0, m1, S1):
    """Matriz A del mapa óptimo T(x) = m1 + A (x - m0)."""
    S0h = np.real(sqrtm(S0)); S0hi = np.linalg.inv(S0h)
    return S0hi @ np.real(sqrtm(S0h @ S1 @ S0h)) @ S0hi

m0, S0 = np.array([0., 0.]), np.array([[1.0, 0.3], [0.3, 0.5]])
m1, S1 = np.array([3., 1.]), np.array([[0.6, -0.4], [-0.4, 1.5]])

w_formula = W2_gauss(m0, S0, m1, S1)
w_pot = ot.gaussian.bures_wasserstein_distance(m0, m1, S0, S1)
print("W_2 fórmula de la traza :", w_formula)
print("W_2 POT (Bures)         :", w_pot)

A = mapa_gauss(m0, S0, m1, S1)
print("\nVerificación A Σ0 A = Σ1 :\n", A @ S0 @ A)
print("A simétrica:", np.allclose(A, A.T), " definida positiva:", np.all(np.linalg.eigvalsh(A) > 0))

In [ ]:
ns = [50, 100, 200, 500, 1000, 2000]
reps = 5
est = np.zeros((len(ns), reps))
for i, n in enumerate(ns):
    for r in range(reps):
        X = rng.multivariate_normal(m0, S0, n)
        Y = rng.multivariate_normal(m1, S1, n)
        est[i, r] = Wp_lp(X, Y, 2)

plt.figure(figsize=(6, 4))
plt.errorbar(ns, est.mean(1), yerr=est.std(1), fmt='o-', capsize=3, label=r'$W_2(\mu_n,\nu_n)$ Monte Carlo')
plt.axhline(w_formula, color='k', ls='--', label='fórmula cerrada')
plt.xscale('log'); plt.xlabel('n muestras'); plt.ylabel(r'$W_2$'); plt.legend(); plt.title('Convergencia de la estimación empírica')
plt.show()

Notar que la estimación empírica **sobreestima** sistemáticamente la distancia para $n$ pequeño: la medida empírica está lejos de la gaussiana que la genera, y ese error se suma. Esto no es un defecto del solver sino un fenómeno estadístico, que en las aplicaciones motiva las regularizaciones entrópicas del último capítulo.

Visualizamos el mapa óptimo afín $T$ sobre una muestra: los puntos de $\mu$ y sus imágenes $T(x)$, que se distribuyen según $\nu$.

In [ ]:
X = rng.multivariate_normal(m0, S0, 400)
TX = m1 + (X - m0) @ A.T
Y = rng.multivariate_normal(m1, S1, 400)

plt.figure(figsize=(6, 5))
plt.scatter(*X.T, s=8, alpha=0.6, label=r'$x\sim\mu$')
plt.scatter(*Y.T, s=8, alpha=0.3, color='gray', label=r'$y\sim\nu$ (independiente)')
plt.scatter(*TX.T, s=8, alpha=0.6, color='C3', label=r'$T(x)$')
for k in range(0, 400, 20):
    plt.plot([X[k, 0], TX[k, 0]], [X[k, 1], TX[k, 1]], color='C3', lw=0.5, alpha=0.6)
plt.axis('equal'); plt.legend(); plt.title(r'Mapa óptimo afín $T(x)=m_1+A(x-m_0)$'); plt.show()

## 3. Comparación entre exponentes

Por la desigualdad de Jensen (o de Hölder), si $1\le q\le p$ entonces

$$
W_q(\mu,\nu)\le W_p(\mu,\nu).
$$

En la dirección opuesta no hay una desigualdad general (piénsese en las medidas con masa que se escapa de la sección siguiente), pero si ambas medidas tienen soporte en un conjunto de diámetro $D$, entonces $|x-y|^p\le D^{p-q}|x-y|^q$ sobre el soporte de cualquier plan, y por lo tanto

$$
W_p(\mu,\nu)^p\le D^{\,p-q}\,W_q(\mu,\nu)^q .
$$

Verificamos ambas desigualdades con medidas discretas aleatorias en $[0,1]^2$ (diámetro $D=\sqrt2$).

In [ ]:
def W_discreta(X, a, Y, b, p):
    M = ot.dist(X, Y, metric='euclidean')**p
    return ot.emd2(a, b, M, numItermax=10_000_000)**(1/p)

n, m = 40, 60
X, Y = rng.random((n, 2)), rng.random((m, 2))
a = rng.dirichlet(np.ones(n)); b = rng.dirichlet(np.ones(m))
D = np.sqrt(2)

ps = [1, 1.5, 2, 3, 4, 6]
W = {p: W_discreta(X, a, Y, b, p) for p in ps}
print("p  ->  W_p"); [print(f"{p:<4} {W[p]:.5f}") for p in ps]

print("\nMonotonía W_q <= W_p (q<p):", all(W[ps[i]] <= W[ps[i+1]] + 1e-12 for i in range(len(ps)-1)))
print("Cota W_p^p <= D^(p-q) W_q^q, con q=1:")
for p in ps[1:]:
    print(f"  p={p}: {W[p]**p:.5f} <= {D**(p-1)*W[1]:.5f}  ->", W[p]**p <= D**(p-1)*W[1] + 1e-12)

## 4. Convergencia en $W_p$, convergencia débil y momentos

El teorema central del capítulo dice que, en $\mathcal P_p(\mathbb R^d)$,

$$
W_p(\mu_n,\mu)\to0
\quad\Longleftrightarrow\quad
\mu_n\rightharpoonup\mu\ \text{ débilmente}\ \ \text{y}\ \ \int|x|^p\,d\mu_n\to\int|x|^p\,d\mu .
$$

La convergencia débil sola **no** alcanza. El ejemplo canónico es una masa pequeña que se escapa al infinito:

$$
\mu_n=\Bigl(1-\frac1n\Bigr)\delta_0+\frac1n\,\delta_{a_n},\qquad a_n\to\infty .
$$

Siempre $\mu_n\rightharpoonup\delta_0$ (la masa en $a_n$ tiende a cero). Pero un cálculo directo da $W_p(\mu_n,\delta_0)^p=\frac1n a_n^p$, de modo que la convergencia en $W_p$ depende de la velocidad de escape **y del exponente**:

- con $a_n=n$: $W_1(\mu_n,\delta_0)=1$ no tiende a cero, y $W_2^2=n\to\infty$;
- con $a_n=\sqrt n$: $W_1=n^{-1/2}\to0$ pero $W_2=1$ no tiende a cero;
- con $a_n=n^{1/4}$: $W_1\to0$ y $W_2\to0$ pero $W_4=1$.

En cada caso lo que falla es exactamente la convergencia del momento de orden $p$: $\int|x|^p\,d\mu_n=\frac1na_n^p$, que debería tender a $\int|x|^p\,d\delta_0=0$. Verificamos con POT que el programa lineal reproduce estos valores.

In [ ]:
def W_escape(n, a_n, p):
    """W_p entre mu_n = (1-1/n) d_0 + (1/n) d_{a_n} y delta_0, vía POT."""
    X = np.array([[0.], [a_n]]); a = np.array([1 - 1/n, 1/n])
    Y = np.array([[0.]]); b = np.array([1.])
    M = ot.dist(X, Y, metric='euclidean')**p
    return ot.emd2(a, b, M)**(1/p)

ns = np.array([2, 5, 10, 20, 50, 100, 200, 500, 1000])
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for k, (nombre, f) in enumerate([("a_n = n", lambda n: n), ("a_n = √n", np.sqrt), ("a_n = n^{1/4}", lambda n: n**0.25)]):
    for p in [1, 2, 4]:
        ax[k].plot(ns, [W_escape(n, f(n), p) for n in ns], 'o-', label=f'$W_{p}$')
    ax[k].set_xscale('log'); ax[k].set_yscale('log'); ax[k].set_title(f'${nombre}$'); ax[k].set_xlabel('n')
    ax[k].axhline(1, color='k', ls=':', lw=0.8)
ax[0].legend(); ax[0].set_ylabel(r'$W_p(\mu_n,\delta_0)$')
plt.suptitle(r'$\mu_n\rightharpoonup\delta_0$ siempre; la convergencia en $W_p$ depende de $p$ y de la velocidad de escape')
plt.tight_layout(); plt.show()

En el otro sentido, cuando **sí** hay convergencia débil con soportes uniformemente acotados (una de las hipótesis bajo las que demostramos la recíproca), la convergencia en $W_p$ es automática para todo $p$. Un ejemplo: la medida uniforme discreta sobre los puntos medios de la grilla, $\mu_n=\frac1n\sum_{k=1}^n\delta_{(k-1/2)/n}$, converge a la uniforme en $[0,1]$, y en dimensión uno podemos calcular $W_p$ exactamente con los cuantiles: $|F_n^{[-1]}(t)-t|\le\frac1{2n}$ para todo $t$, luego $W_p(\mu_n,U[0,1])\le\frac1{2n}$ para todo $p$, y de hecho $W_p^p=\int_0^1|F_n^{[-1]}(t)-t|^p\,dt=\frac{1}{(p+1)(2n)^p}$.

In [ ]:
def Wp_grilla_vs_uniforme(n, p, K=200_001):
    t = np.linspace(0, 1, K)[1:-1]
    Finv_n = (np.floor(t*n) + 0.5)/n        # pseudoinversa de la uniforme en {(k-1/2)/n}: escalera
    Finv_U = t                              # pseudoinversa de U[0,1]
    return np.mean(np.abs(Finv_n - Finv_U)**p)**(1/p)

print(f"{'n':>6} {'W_1 numérico':>14} {'exacto':>10} {'W_2 numérico':>14} {'exacto':>10}")
for n in [5, 10, 50, 100, 500]:
    e1 = 1/(2*2*n); e2 = (1/(3*(2*n)**2))**0.5
    print(f"{n:>6} {Wp_grilla_vs_uniforme(n,1):>14.6f} {e1:>10.6f} {Wp_grilla_vs_uniforme(n,2):>14.6f} {e2:>10.6f}")

## 5. Interpolación por desplazamiento

Si $\pi$ es un plan óptimo para $W_2$ entre $\mu$ y $\nu$, la curva

$$
\mu_t=(e_t)_\#\pi,\qquad e_t(x,y)=(1-t)x+ty,\qquad t\in[0,1],
$$

es una **geodésica** en $(\mathcal P_2,W_2)$: $W_2(\mu_s,\mu_t)=|t-s|\,W_2(\mu,\nu)$. Cuando el plan es inducido por un mapa $T$ (Brenier), $\mu_t=\bigl((1-t)\,\mathrm{id}+tT\bigr)_\#\mu$: **cada partícula se mueve en línea recta** de $x$ a $T(x)$ a velocidad constante. Esto es lo que McCann llamó interpolación por desplazamiento, y contrasta con la interpolación lineal $(1-t)\mu+t\nu$, en la que la masa no se mueve: simplemente se desvanece en un lado y aparece en el otro.

### 5.1 En dimensión uno

Comparamos las dos interpolaciones entre dos mezclas de gaussianas. En dimensión uno $T=T_{\mathrm{mon}}$, y para muestras de igual tamaño el mapa monótono simplemente empareja los datos ordenados.

In [ ]:
n = 4000
mu_s = np.concatenate([rng.normal(-3, 0.5, n//2), rng.normal(-1, 0.4, n//2)])
nu_s = np.concatenate([rng.normal(2, 0.7, n//4), rng.normal(4, 0.3, 3*n//4)])
xs, ys = np.sort(mu_s), np.sort(nu_s)      # T_mon: x_(k) -> y_(k)

ts = [0, 0.25, 0.5, 0.75, 1]
bins = np.linspace(-5, 6, 120)
fig, ax = plt.subplots(2, len(ts), figsize=(15, 5), sharex=True, sharey=True)
for k, t in enumerate(ts):
    # desplazamiento: partícula k está en (1-t) x_(k) + t y_(k)
    ax[0, k].hist((1-t)*xs + t*ys, bins=bins, density=True, color='C3', alpha=0.8)
    ax[0, k].set_title(f't = {t}')
    # lineal: mezcla de mu y nu con pesos (1-t), t
    mezcla = np.concatenate([rng.choice(mu_s, int((1-t)*n)), rng.choice(nu_s, n - int((1-t)*n))])
    ax[1, k].hist(mezcla, bins=bins, density=True, color='C0', alpha=0.8)
ax[0, 0].set_ylabel('desplazamiento\n' + r'$((1-t)\,\mathrm{id}+tT)_\#\mu$'); ax[1, 0].set_ylabel('lineal\n' + r'$(1-t)\mu+t\nu$')
plt.tight_layout(); plt.show()

En la fila superior la masa **viaja**; en la inferior se desvanece y reaparece. Verificamos ahora la propiedad geodésica: $W_2(\mu_s,\mu_t)=|t-s|\,W_2(\mu,\nu)$, mientras que la interpolación lineal **no** es geodésica: la distancia $W_2\bigl((1-s)\mu+s\nu,(1-t)\mu+t\nu\bigr)$ se comporta como $\sqrt{|t-s|}$ (mover una fracción $|t-s|$ de la masa de $\mu$ hasta $\nu$ cuesta $|t-s|\,W_2^2$ en costo cuadrático), que para $|t-s|$ pequeño es mucho **mayor** que $|t-s|\,W_2(\mu,\nu)$. De hecho la curva lineal tiene longitud infinita (Ejercicio 3).

In [ ]:
W_total = Wp_1d(xs, ys, 2)
print(f"W_2(mu, nu) = {W_total:.4f}\n")
print(f"{'(s,t)':>12} {'|t-s| W_2':>10} {'desplaz.':>10} {'lineal':>10}")
sub = rng.choice(n, 800, replace=False)     # submuestra para el LP de la interpolación lineal
for s, t in [(0, 0.5), (0.25, 0.75), (0.5, 1), (0.2, 0.3)]:
    d_desp = Wp_1d((1-s)*xs + s*ys, (1-t)*xs + t*ys, 2)
    # lineal: medidas con pesos (no equiponderadas) sobre la unión de soportes -> LP
    Xu = np.concatenate([mu_s[sub], nu_s[sub]])
    ws = np.concatenate([np.full(800, (1-s)/800), np.full(800, s/800)])
    wt = np.concatenate([np.full(800, (1-t)/800), np.full(800, t/800)])
    M = ot.dist(Xu[:, None], Xu[:, None])            # métrica por defecto: euclídea al cuadrado
    d_lin = np.sqrt(ot.emd2(ws, wt, M, numItermax=10_000_000))
    print(f"({s:.2f},{t:.2f}) {abs(t-s)*W_total:>10.4f} {d_desp:>10.4f} {d_lin:>10.4f}")

### 5.2 Gaussianas en el plano

Entre gaussianas la interpolación por desplazamiento permanece gaussiana: como $T(x)=m_1+A(x-m_0)$ es afín, $\mu_t=N(m_t,\Sigma_t)$ con

$$
m_t=(1-t)m_0+tm_1,\qquad \Sigma_t=\bigl((1-t)I+tA\bigr)\,\Sigma_0\,\bigl((1-t)I+tA\bigr).
$$

Dibujamos las elipses de confianza de $\mu_t$ y verificamos la propiedad geodésica con la fórmula cerrada de la Sección 2.

In [ ]:
def elipse(m, S, ax, **kw):
    w, V = np.linalg.eigh(S)
    th = np.linspace(0, 2*np.pi, 200)
    pts = (V * np.sqrt(w)) @ np.vstack([np.cos(th), np.sin(th)]) * 2   # 2 desvíos
    ax.plot(m[0] + pts[0], m[1] + pts[1], **kw)

I = np.eye(2)
fig, ax = plt.subplots(figsize=(7, 5))
for t in np.linspace(0, 1, 6):
    Bt = (1-t)*I + t*A
    mt, St = (1-t)*m0 + t*m1, Bt @ S0 @ Bt
    elipse(mt, St, ax, color=plt.cm.viridis(t), lw=2, label=f't={t:.1f}')
    ax.plot(*mt, 'o', color=plt.cm.viridis(t))
ax.axis('equal'); ax.legend(); ax.set_title(r'Geodésica de $W_2$ entre dos gaussianas'); plt.show()

print("Verificación de W_2(mu_s, mu_t) = |t-s| W_2(mu_0, mu_1):")
for s, t in [(0, 0.5), (0.3, 0.8), (0.5, 1)]:
    Bs, Bt = (1-s)*I + s*A, (1-t)*I + t*A
    d = W2_gauss((1-s)*m0 + s*m1, Bs @ S0 @ Bs, (1-t)*m0 + t*m1, Bt @ S0 @ Bt)
    print(f"  (s,t)=({s},{t}):  {d:.6f}  vs  {abs(t-s)*w_formula:.6f}")

## 6. Dualidad de Kantorovich–Rubinstein

Para $p=1$ el teorema de dualidad toma una forma particularmente limpia:

$$
W_1(\mu,\nu)=\sup\Bigl\{\int\varphi\,d(\mu-\nu):\ \varphi\ \text{1-Lipschitz}\Bigr\}.
$$

Para medidas discretas sobre un conjunto finito de puntos $\{z_k\}$ (la unión de los dos soportes), el supremo se puede calcular como un programa lineal en las variables $\varphi_k=\varphi(z_k)$: maximizar $\sum_k\varphi_k(\mu_k-\nu_k)$ sujeto a $|\varphi_k-\varphi_l|\le|z_k-z_l|$ para todo par $k,l$. (Toda función 1-Lipschitz sobre un subconjunto de $\mathbb R^d$ se extiende a una 1-Lipschitz en $\mathbb R^d$ —extensión de McShane—, así que restringirse a los valores en los puntos no pierde nada.)

Resolvemos ese programa lineal con `scipy` y comparamos con el valor primal que calcula POT: el teorema afirma que coinciden.

In [ ]:
# medidas discretas en el plano con soportes distintos
n, m = 12, 15
X, Y = rng.random((n, 2))*3, rng.random((m, 2))*3 + np.array([1.0, 0.5])
a, b = rng.dirichlet(np.ones(n)), rng.dirichlet(np.ones(m))

# primal
M = ot.dist(X, Y, metric='euclidean')
W1_primal = ot.emd2(a, b, M)

# dual K-R: variables phi_k en Z = X ∪ Y ; maximizar sum phi_k (mu_k - nu_k)
Z = np.vstack([X, Y]); N = len(Z)
diff = np.concatenate([a, -b])                     # mu - nu como vector sobre Z
Dz = ot.dist(Z, Z, metric='euclidean')
rows, rhs = [], []
for k in range(N):
    for l in range(N):
        if k != l:
            r = np.zeros(N); r[k], r[l] = 1, -1     # phi_k - phi_l <= |z_k - z_l|
            rows.append(r); rhs.append(Dz[k, l])
res = linprog(-diff, A_ub=np.array(rows), b_ub=np.array(rhs), bounds=[(None, None)]*N, method='highs')
phi = res.x
W1_dual = diff @ phi

print("W_1 primal (POT)          :", W1_primal)
print("W_1 dual Kantorovich-Rubinstein:", W1_dual)
print("phi es 1-Lipschitz sobre Z:", np.all(np.abs(phi[:, None] - phi[None, :]) <= Dz + 1e-9))

In [ ]:
# dibujamos el potencial 1-Lipschitz óptimo y el plan óptimo
P = ot.emd(a, b, M)
fig, ax = plt.subplots(figsize=(7, 5.5))
sc = ax.scatter(*Z.T, c=phi, cmap='coolwarm', s=400*np.concatenate([a, b]), edgecolor='k', zorder=3)
for i in range(n):
    for j in range(m):
        if P[i, j] > 1e-10:
            ax.plot([X[i, 0], Y[j, 0]], [X[i, 1], Y[j, 1]], color='gray', lw=8*P[i, j]/P.max(), alpha=0.6)
ax.scatter(*X.T, marker='o', facecolor='none', edgecolor='C0', s=200, lw=2, label=r'sop $\mu$')
ax.scatter(*Y.T, marker='s', facecolor='none', edgecolor='C1', s=200, lw=2, label=r'sop $\nu$')
plt.colorbar(sc, label=r'$\varphi$ (1-Lipschitz óptimo)')
ax.axis('equal'); ax.legend(); ax.set_title(r'Plan óptimo (grises) y potencial de Kantorovich–Rubinstein (color)')
plt.show()

Observar que $\varphi$ **decrece a lo largo de cada segmento activo del plan**: si $\pi_{ij}>0$ entonces $\varphi(x_i)-\varphi(y_j)=|x_i-y_j|$, es decir, sobre el soporte del plan la desigualdad de Lipschitz se satura. Es la condición de holgura complementaria del programa lineal, y es la versión $p=1$ del criterio de optimalidad $\varphi(x)+\psi(y)=c(x,y)$ en el soporte, con $\psi=\varphi^c=-\varphi$.

In [ ]:
sat = [(i, j, phi[i] - phi[n + j], M[i, j]) for i in range(n) for j in range(m) if P[i, j] > 1e-10]
print(f"{'i':>3} {'j':>3} {'phi(x_i)-phi(y_j)':>18} {'|x_i-y_j|':>10}")
for i, j, d, c in sat[:10]:
    print(f"{i:>3} {j:>3} {d:>18.6f} {c:>10.6f}")
print("...\nSaturación en todo el soporte del plan:", all(abs(d - c) < 1e-7 for *_, d, c in sat))

## Ejercicios computacionales

Los enunciados siguientes figuran también en la sección de ejercicios del capítulo correspondiente de las notas.

1. **Escala.** Probar numéricamente (y luego a mano) que $W_p(\lambda_\#\mu,\lambda_\#\nu)=\lambda\,W_p(\mu,\nu)$ para la dilatación $x\mapsto\lambda x$, y que $W_p$ es invariante por traslaciones simultáneas.

2. **Traslaciones.** Para $\nu=\tau_{v\,\#}\mu$ (trasladar $\mu$ por el vector $v$) se tiene $W_p(\mu,\nu)\le|v|$. Probar que para $p=2$ hay siempre igualdad, y verificarlo numéricamente con medidas discretas. (Sugerencia: probar primero la descomposición $W_2^2(\mu,\nu)=|m_\mu-m_\nu|^2+W_2^2(\tilde\mu,\tilde\nu)$, donde $m$ denota la media y $\tilde\mu,\tilde\nu$ las medidas centradas.)

3. **Geodésicas lineales.** Calcular la longitud de la curva $t\mapsto(1-t)\mu+t\nu$ en $(\mathcal P_2,W_2)$ para $\mu=\delta_0$, $\nu=\delta_1$ en $\mathbb R$: mostrar que $W_2\bigl((1-s)\mu+s\nu,(1-t)\mu+t\nu\bigr)=\sqrt{|t-s|}$ y deducir que la longitud es infinita. Comparar con la Sección 5.1.

4. **Un tercer exponente.** Extender la Sección 4 para exhibir, para cada par $1\le q<p$, una sucesión $\mu_n$ con $W_q(\mu_n,\delta_0)\to0$ pero $W_p(\mu_n,\delta_0)\to\infty$.

5. **K–R con `ot.emd`.** POT devuelve también los potenciales duales (`ot.emd(a, b, M, log=True)`). Comparar los potenciales $(u,v)$ obtenidos allí con el $\varphi$ de la Sección 6: no tienen por qué coincidir (¿por qué?), pero el valor dual $\sum_ia_iu_i+\sum_jb_jv_j$ debe ser $W_1$. ¿Es $v=-u$ sobre los puntos comunes? ¿Por qué no necesariamente?